# Recommendation System - Training & Benchmark

This notebook trains and benchmarks V5 (LSTM baseline) and V5.1 (LSTM+Attention layer) models for product recommendations.

## Contents
1. Setup & Imports
2. Data Loading
3. Data Preparation
4. V5 Model Training (LSTM without Attention)
5. V5.1 Model Training (LSTM + Attention)
6. Cross-Validation (3-Fold)
7. Results Comparison

In [1]:
# @hide
# Setup
import os
import sys
import numpy as np
import pickle
from collections import defaultdict, Counter
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")
import django

os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

import sys
import django
from unittest.mock import MagicMock

try:
    import cv2
except ImportError:
    sys.modules['cv2'] = MagicMock()

project_root = '/Users/alifffayruz/Documents/UWE/25-26 Semester/DaESD/code/DESD-BRFN/DESD_BRFN'
os.chdir(project_root)
sys.path.insert(0, project_root)
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'BRFN.settings')

from django.conf import settings

# if not django.apps.apps.ready:
django.setup()

if hasattr(settings, 'DATABASES'):
    if settings.DATABASES['default']['HOST'] == 'db':
        settings.DATABASES['default']['HOST'] = 'localhost'
        print("Database host changed from 'db' to 'localhost'")

print("Django setup complete")

# TensorFlow
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.cluster import KMeans
from sklearn.model_selection import KFold

# Project imports
from orders.models import OrderItem
from products.models import Product

Database host changed from 'db' to 'localhost'
Django setup complete


In [2]:
# @hide
# Constants
BATCH_SIZE = 64
EPOCHS = 30
LEARNING_RATE_V5 = 5e-4
LEARNING_RATE_V5_1 = 1e-3
MAX_ORDER_HISTORY = 15
MAX_ITEMS_PER_ORDER = 5
VOCAB_SIZE = 200
NUM_CLASSES = 160
NUM_NEGATIVES = 20
DROPOUT = 0.2
N_FOLDS = 5
SEED = 42

## 1. Data Loading

In [3]:
def extract_temporal_features(timestamp):
    """Extract temporal features from datetime."""
    day_of_week = timestamp.weekday()
    day_sin = np.sin(2 * np.pi * day_of_week / 7.0)
    day_cos = np.cos(2 * np.pi * day_of_week / 7.0)
    month = timestamp.month
    month_sin = np.sin(2 * np.pi * month / 12.0)
    month_cos = np.cos(2 * np.pi * month / 12.0)
    is_weekend = 1.0 if day_of_week >= 5 else 0.0
    return [day_sin, day_cos, month_sin, month_cos, is_weekend]


def get_user_orders():
    """Load user orders from database."""
    user_orders = defaultdict(list)
    order_items_query = OrderItem.objects.filter(
        producer_order__payment__payment_status="paid"
    ).exclude(
        producer_order__payment__user__customer_profile__id__range=(1,5)
    ).select_related("product", "producer_order__payment__user"
    ).order_by("producer_order__payment__created_at")
    
    print(f"[DEBUG] Raw OrderItem query count: {order_items_query.count()}")
    
    order_items = list(order_items_query)
    print(f"[DEBUG] Order items loaded: {len(order_items)}")
    
    current_user, current_order, products = None, None, []
    for item in order_items:
        user = item.producer_order.payment.user
        order_id = item.producer_order.payment.id
        timestamp = item.producer_order.payment.created_at
        product_id = item.product.id
        
        if user is None:
            continue
        if current_user != user.id:
            if current_user and current_order and products:
                user_orders[current_user].append((current_order, timestamp, list(set(products))))
            current_user, current_order, products = user.id, None, []
        if current_order != order_id:
            if current_order and products:
                user_orders[current_user].append((current_order, timestamp, list(set(products))))
            current_order, products = order_id, [product_id]
        else:
            if product_id not in products:
                products.append(product_id)
    if current_user and current_order and products:
        user_orders[current_user].append((current_order, timestamp, products))
    
    print(f"[DEBUG] Unique users: {len(user_orders)}")
    return dict(user_orders)

In [4]:
# Load data
print("=" * 50)
print("Loading order data...")
print("=" * 50)
user_orders = get_user_orders()
print(f"\nUsers loaded: {len(user_orders)}")
total_orders = sum(len(orders) for orders in user_orders.values())
print(f"Total orders: {total_orders}")
users_with_2plus = sum(1 for orders in user_orders.values() if len(orders) >= 2)
print(f"Users with 2+ orders: {users_with_2plus}")

Loading order data...
[DEBUG] Raw OrderItem query count: 13625
[DEBUG] Order items loaded: 13625
[DEBUG] Unique users: 95

Users loaded: 95
Total orders: 4546
Users with 2+ orders: 95


## 2. Data Preparation

In [5]:
def build_product_mappings(user_orders, num_products=160):
    """Build product-to-index and index-to-product mappings."""
    counter = Counter()
    for orders in user_orders.values():
        for _, _, ps in orders:
            counter.update(ps)
    
    top_products = {p for p, _ in counter.most_common(num_products)}
    product_to_idx = {p: i+2 for i, p in enumerate(sorted(top_products))}
    product_to_idx[0] = 0  # padding
    idx_to_product = {i+2: p for i, p in enumerate(sorted(top_products))}
    
    return product_to_idx, idx_to_product


def build_sequences(user_orders, product_to_idx, other_token, all_pids,
              max_hist=15, max_items=5, num_neg=20, seed=42):
    """Build training sequences with negative sampling."""
    np.random.seed(seed)
    X_prod, X_time, y_labels, user_ids = [], [], [], []
    all_pids = list(all_pids)
    
    for uid, orders in user_orders.items():
        if len(orders) < 2:
            continue
        for i in range(len(orders) - 1):
            ctx = orders[:i+1]
            tgt = orders[i+1]
            
            ctx_prods, ctx_times = [], []
            for _, ts, ps in ctx[-max_hist:]:
                ps = ps[:max_items]
                if len(ps) < max_items:
                    ps = ps + [0] * (max_items - len(ps))
                enc = [product_to_idx.get(p, other_token) if p != 0 else 0 for p in ps]
                ctx_prods.append(enc)
                ctx_times.append(extract_temporal_features(ts))
            
            while len(ctx_prods) < max_hist:
                ctx_prods = [[0]*max_items] + ctx_prods
                ctx_times = [[0.0]*5] + ctx_times
            
            # Positive products
            pos_idx = [product_to_idx.get(p, other_token) for p in tgt[2] if p in product_to_idx]
            pos_idx = [p for p in pos_idx if p >= 2]
            if not pos_idx:
                continue
            
            # Negative sampling
            neg_pool = [p for p in all_pids if p not in tgt[2]]
            if len(neg_pool) < 1:
                continue
            
            for pidx in pos_idx:
                X_prod.append(ctx_prods)
                X_time.append(ctx_times)
                y_labels.append(pidx)
                user_ids.append(uid)
    
    X_prod = np.array(X_prod, dtype=np.int32)
    X_time = np.array(X_time, dtype=np.float32)
    y_labels = np.array(y_labels, dtype=np.int32)
    
    print(f"[DEBUG] Samples: {len(X_prod)}")
    return X_prod, X_time, y_labels, user_ids


def get_clusters(user_orders, product_to_idx, n_clusters=8):
    """Cluster users based on purchase patterns."""
    features = {}
    for uid, orders in user_orders.items():
        if len(orders) < 3:
            continue
        cnt = Counter()
        for _, _, ps in orders:
            cnt.update(ps)
        arr = np.zeros(max(product_to_idx.values()) + 1, dtype=np.float32)
        for p, c in cnt.items():
            i = product_to_idx.get(p)
            if i and i < len(arr):
                arr[i] = c
        if arr.sum() > 0:
            arr = arr / arr.sum()
        features[uid] = arr
    
    if len(features) < n_clusters:
        return {u: 0 for u in features}, 1
    
    X = np.array(list(features.values()))
    km = KMeans(min(n_clusters, len(X)), random_state=42, n_init=3)
    labels = km.fit_predict(X)
    return {u: labels[i] for i, u in enumerate(sorted(features.keys()))}, len(set(labels))

In [6]:
# Build mappings and sequences
print("Building product mappings...")
product_to_idx, idx_to_product = build_product_mappings(user_orders, NUM_CLASSES)
print(f"Vocabulary size: {max(product_to_idx.values()) + 1}")
all_pids = set(product_to_idx.keys())
all_pids.discard(0)

print("\nBuilding sequences...")
Xp, Xt, y, uids = build_sequences(
    user_orders, product_to_idx, other_token=1, all_pids=all_pids,
    max_hist=MAX_ORDER_HISTORY, max_items=MAX_ITEMS_PER_ORDER,
    num_neg=NUM_NEGATIVES, seed=SEED
)
print(f"Training samples: {len(Xp)}")
print(f"Product shape: {Xp.shape}")
print(f"Temporal shape: {Xt.shape}")

print("\nBuilding user clusters...")
user_to_cluster, n_clusters = get_clusters(user_orders, product_to_idx)
print(f"Number of clusters: {n_clusters}")

Building product mappings...
Vocabulary size: 162

Building sequences...
[DEBUG] Samples: 8307
Training samples: 8307
Product shape: (8307, 15, 5)
Temporal shape: (8307, 15, 5)

Building user clusters...
Number of clusters: 8


## 3. V5 Model (LSTM without Attention)

In [7]:
def build_model_v5(num_classes, vocab_size, max_hist=15, max_items=5, lstm_units=64, dropout=0.2):
    """Build V5 model: LSTM without attention."""
    product_ids = keras.Input(shape=(max_hist, max_items), name="product_input")
    temporal = keras.Input(shape=(max_hist, 5), name="time_input")
    user_cluster = keras.Input(shape=(1,), name="user_cluster")
    
    # Embedding
    emb = layers.Embedding(vocab_size, 32, mask_zero=True)(product_ids)
    flat_emb = layers.Reshape((max_hist, max_items * 32))(emb)
    combined = layers.Concatenate()([flat_emb, temporal])
    
    # LSTM (no attention - just last hidden state)
    lstm_out = layers.LSTM(lstm_units, return_sequences=False)(combined)
    lstm_out = layers.Dropout(dropout)(lstm_out)
    
    # Cluster embedding
    c_emb = layers.Embedding(10, 8)(user_cluster)
    merged = layers.Concatenate()([lstm_out, layers.Flatten()(c_emb)])
    
    x = layers.Dense(64, activation="relu")(merged)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(32, activation="relu")(x)
    x = layers.Dropout(dropout * 0.5)(x)
    logits = layers.Dense(num_classes, name="logits")(x)
    
    model = keras.Model(inputs=[product_ids, temporal, user_cluster], outputs=logits)
    model.compile(
        optimizer=keras.optimizers.Adam(LEARNING_RATE_V5),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=[keras.metrics.SparseTopKCategoricalAccuracy(k=5)]
    )
    return model

In [8]:
# Train V5 model
print("=" * 50)
print("Training V5 Model (LSTM without Attention)")
print("=" * 50)

# Split data
n_samples = len(Xp)
indices = np.arange(n_samples)
np.random.seed(SEED)
np.random.shuffle(indices)

split_idx = int(n_samples * 0.8)
train_idx = indices[:split_idx]
val_idx = indices[split_idx:]

Xp_train, Xp_val = Xp[train_idx], Xp[val_idx]
Xt_train, Xt_val = Xt[train_idx], Xt[val_idx]
y_train, y_val = y[train_idx], y[val_idx]

c_train = np.array([[user_to_cluster.get(uids[i], 0)] for i in train_idx], dtype=np.int32)
c_val = np.array([[user_to_cluster.get(uids[i], 0)] for i in val_idx], dtype=np.int32)

print(f"Train samples: {len(Xp_train)}, Val samples: {len(Xp_val)}")

# Build and train model
model_v5 = build_model_v5(
    num_classes=VOCAB_SIZE,
    vocab_size=VOCAB_SIZE,
    max_hist=MAX_ORDER_HISTORY,
    max_items=MAX_ITEMS_PER_ORDER,
    lstm_units=64,
    dropout=DROPOUT
)
model_v5.summary()

# Train
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', 
    patience=5, 
    min_delta=0.001,
    restore_best_weights=True
)

history_v5 = model_v5.fit(
    [Xp_train, Xt_train, c_train], 
    y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=([Xp_val, Xt_val, c_val], y_val),
    callbacks=[early_stop],
    verbose=1
)

Training V5 Model (LSTM without Attention)
Train samples: 6645, Val samples: 1662


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ product_input       │ (None, 15, 5)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 15, 5, 32) │      6,400 │ product_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 15, 160)   │          0 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_input          │ (None, 15, 5)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 15, 165)   │          0 │ reshape[0][0],    │
│ (Concatenate)       │                   │            │ time_input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_cluster        │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 64)        │     58,880 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 1, 8)      │         80 │ user_cluster[0][… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 64)        │          0 │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 8)         │          0 │ embedding_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 72)        │          0 │ dropout[0][0],    │
│ (Concatenate)       │                   │            │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │      4,672 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 64)        │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 32)        │      2,080 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 32)        │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ logits (Dense)      │ (None, 200)       │      6,600 │ dropout_2[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 78,712 (307.47 KB)

 Trainable params: 78,712 (307.47 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
104/104 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 5.2635 - sparse_top_k_categorical_accuracy: 0.0307 - val_loss: 5.1892 - val_sparse_top_k_categorical_accuracy: 0.0337
Epoch 2/30
104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 5.1433 - sparse_top_k_categorical_accuracy: 0.0408 - val_loss: 5.1164 - val_sparse_top_k_categorical_accuracy: 0.0361
Epoch 3/30
104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 5.0987 - sparse_top_k_categorical_accuracy: 0.0462 - val_loss: 5.0971 - val_sparse_top_k_categorical_accuracy: 0.0415
Epoch 4/30
104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 5.0862 - sparse_top_k_categorical_accuracy: 0.0408 - val_loss: 5.0914 - val_sparse_top_k_categorical_accuracy: 0.0361
Epoch 5/30
104/104 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 5.0802 - sparse_top_k_categorical_accuracy: 0.0471 - val_loss: 5.0869 - val_sparse_top_k_categorical_accuracy: 0.0439
Epoch 6/30
104/104 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 5.0708 - sparse_top_k_categorical_accuracy: 0.0492

## 4. V5.1 Model (LSTM + Attention)

In [9]:
def build_model_v5_1(num_classes, vocab_size, max_hist=15, max_items=5, lstm_units=64, n_cluster=8, dropout=0.2):
    """Build V5.1 model: LSTM with dot-product attention."""
    product_ids = keras.Input(shape=(max_hist, max_items,), name="product_input")
    temporal = keras.Input(shape=(max_hist, 5), name="time_input")
    user_cluster = keras.Input(shape=(1,), name="user_cluster")
    
    # Embedding
    emb = layers.Embedding(vocab_size, 32, mask_zero=True)(product_ids)
    flat_emb = layers.Reshape((max_hist, max_items * 32))(emb)
    
    # Concatenate with temporal
    combined = layers.Concatenate()([flat_emb, temporal])
    
    # LSTM
    lstm_out = layers.LSTM(lstm_units, return_sequences=True)(combined)
    lstm_out = layers.Dropout(dropout)(lstm_out)
    
    # Attention: focus on last hidden state
    last_hidden = layers.Reshape((1, lstm_units))(lstm_out[:, -1, :])
    
    attn_scores = layers.Dot(axes=(2, 2))([last_hidden, lstm_out])
    attn_weights = layers.Softmax(name="attention")(attn_scores)
    attn_weights_flat = layers.Reshape((max_hist,), name="attention_flat")(attn_weights)
    
    # Context vector
    context = layers.Dot(axes=(1, 1))([attn_weights_flat, lstm_out])
    context = layers.Dropout(dropout)(context)
    
    # Cluster embedding
    c_emb = layers.Embedding(n_cluster + 1, 8)(user_cluster)
    
    # Merge
    merged = layers.Concatenate()([context, layers.Flatten()(c_emb)])
    
    x = layers.Dense(64, activation="relu")(merged)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(32, activation="relu")(x)
    x = layers.Dropout(dropout * 0.5)(x)
    
    logits = layers.Dense(num_classes + 2, activation=None, name="logits")(x)
    
    model = keras.Model(
        inputs=[product_ids, temporal, user_cluster], 
        outputs=[logits, attn_weights_flat]
    )
    model.compile(
        optimizer=keras.optimizers.Adam(LEARNING_RATE_V5_1),
        loss={
            "logits": keras.losses.SparseCategoricalCrossentropy(from_logits=True),
            "attention_flat": None
        }
    )
    return model

In [10]:
# Train V5.1 model
print("=" * 50)
print("Training V5.1 Model (LSTM + Attention)")
print("=" * 50)

# Build and train model
model_v5_1 = build_model_v5_1(
    num_classes=NUM_CLASSES,
    vocab_size=VOCAB_SIZE,
    max_hist=MAX_ORDER_HISTORY,
    max_items=MAX_ITEMS_PER_ORDER,
    lstm_units=64,
    n_cluster=n_clusters,
    dropout=DROPOUT
)
model_v5_1.summary()

# Train
attn_dummy = np.zeros((len(y_train), MAX_ORDER_HISTORY), dtype=np.float32)
attn_dummy_val = np.zeros((len(y_val), MAX_ORDER_HISTORY), dtype=np.float32)

early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', 
    patience=5, 
    min_delta=0.001,
    restore_best_weights=True
)

history_v5_1 = model_v5_1.fit(
    [Xp_train, Xt_train, c_train], 
    {"logits": y_train, "attention_flat": attn_dummy},
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=([Xp_val, Xt_val, c_val], {"logits": y_val, "attention_flat": attn_dummy_val}),
    callbacks=[early_stop],
    verbose=1
)

Training V5.1 Model (LSTM + Attention)


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ product_input       │ (None, 15, 5)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, 15, 5, 32) │      6,400 │ product_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_1 (Reshape) │ (None, 15, 160)   │          0 │ embedding_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_input          │ (None, 15, 5)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_2       │ (None, 15, 165)   │          0 │ reshape_1[0][0],  │
│ (Concatenate)       │                   │            │ time_input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 15, 64)    │     58,880 │ concatenate_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 15, 64)    │          0 │ lstm_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item (GetItem)  │ (None, 64)        │          0 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_2 (Reshape) │ (None, 1, 64)     │          0 │ get_item[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dot (Dot)           │ (None, 1, 15)     │          0 │ reshape_2[0][0],  │
│                     │                   │            │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention (Softmax) │ (None, 1, 15)     │          0 │ dot[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_flat      │ (None, 15)        │          0 │ attention[0][0]   │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_cluster        │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dot_1 (Dot)         │ (None, 64)        │          0 │ attention_flat[0… │
│                     │                   │            │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_3         │ (None, 1, 8)      │         72 │ user_cluster[0][… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 64)        │          0 │ dot_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 8)         │          0 │ embedding_3[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_3       │ (None, 72)        │          0 │ dropout_4[0][0],  │
│ (Concatenate)       │                   │            │ flatten_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64)        │      4,672 │ concatenate_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 64)        │          0 │ dense_2[0][0]   

 Total params: 77,450 (302.54 KB)

 Trainable params: 77,450 (302.54 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
104/104 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 5.0855 - val_loss: 5.0822
Epoch 2/30
104/104 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 5.0741 - val_loss: 5.0772
Epoch 3/30
104/104 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 5.0624 - val_loss: 5.0745
Epoch 4/30
104/104 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 5.0499 - val_loss: 5.0664
Epoch 5/30
104/104 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 5.0353 - val_loss: 5.0592
Epoch 6/30
104/104 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 5.0154 - val_loss: 5.0531
Epoch 7/30
104/104 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 4.9863 - val_loss: 5.0392
Epoch 8/30
104/104 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 4.9590 - val_loss: 5.0369
Epoch 9/30
104/104 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 4.9310 - val_loss: 5.0230
Epoch 10/30
104/104 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 4.8956 - val_loss: 5.0092
Epoch 11/30
104/104 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 4.8623 - val_loss: 4.9970
Epoch 12/30
104/104 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step

## 5. Save Models

In [11]:
# Save models
output_dir = "ml/recommendation"
os.makedirs(output_dir, exist_ok=True)

print("Saving models...")
model_v5.save(f"{output_dir}/sigmoid_v5.keras")
model_v5_1.save(f"{output_dir}/sigmoid_v5_1.keras")

mappings = {
    "p2i": product_to_idx,
    "i2p": idx_to_product,
    "u2c": user_to_cluster,
    "max_items": MAX_ITEMS_PER_ORDER,
    "max_orders": MAX_ORDER_HISTORY,
}

with open(f"{output_dir}/sigmoid_v5_mappings.pkl", "wb") as f:
    pickle.dump(mappings, f)
with open(f"{output_dir}/sigmoid_v5_1_mappings.pkl", "wb") as f:
    pickle.dump(mappings, f)

print(f"Models saved to {output_dir}/")

Saving models...
Models saved to ml/recommendation/


## 6. Evaluation Metrics

In [12]:
def evaluate_model(model, Xp, Xt, y_true, c, model_type="v5", top_k=10):
    """Evaluate model on test data."""
    if model_type == "v5":
        logits = model.predict([Xp, Xt, c], verbose=0)
        probs = tf.nn.softmax(logits, axis=-1).numpy()
    else:
        logits, _ = model.predict([Xp, Xt, c], verbose=0)
        probs = tf.nn.softmax(logits, axis=-1).numpy()
    
    hit1 = hit3 = hit5 = hit10 = 0
    mrr = 0
    total = len(y_true)
    
    for i, true_idx in enumerate(y_true):
        top_indices = np.argsort(probs[i])[-top_k:]
        
        if true_idx in top_indices[-1:]:
            hit1 += 1
        if true_idx in top_indices[-3:]:
            hit3 += 1
        if true_idx in top_indices[-5:]:
            hit5 += 1
        if true_idx in top_indices:
            hit10 += 1
        
        # MRR
        for rank, idx in enumerate(reversed(top_indices), 1):
            if idx == true_idx:
                mrr += 1.0 / rank
                break
    
    return {
    "hit@1": hit1 / total,
    "hit@3": hit3 / total,
    "hit@5": hit5 / total,
    "hit@10": hit10 / total,
    "mrr": mrr / total,
    "n_samples": total
}

## 7. Cross-Validation (K-Fold)

In [13]:
print("=" * 60)
print("K-Fold Cross-Validation")
print("=" * 60)

# Setup KFold
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
n_samples = len(Xp)

v5_fold_results = []
v5_1_fold_results = []

for fold, (train_idx, val_idx) in enumerate(kf.split(np.arange(n_samples))):
    print(f"\n--- Fold {fold + 1}/{N_FOLDS} ---")
    print(f"Train: {len(train_idx)}, Val: {len(val_idx)}")
    
    Xp_train, Xp_val = Xp[train_idx], Xp[val_idx]
    Xt_train, Xt_val = Xt[train_idx], Xt[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    c_train = np.array([[user_to_cluster.get(uids[i], 0)] for i in train_idx], dtype=np.int32)
    c_val = np.array([[user_to_cluster.get(uids[i], 0)] for i in val_idx], dtype=np.int32)
    
    attn_dummy = np.zeros((len(y_val), MAX_ORDER_HISTORY), dtype=np.float32)
    
    # Train V5
    print("Training V5...")
    model_v5_cv = build_model_v5(
        num_classes=VOCAB_SIZE, vocab_size=VOCAB_SIZE,
        max_hist=MAX_ORDER_HISTORY, max_items=MAX_ITEMS_PER_ORDER,
        lstm_units=64, dropout=DROPOUT
    )
    early_stop = keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=3, restore_best_weights=True
    )
    model_v5_cv.fit(
        [Xp_train, Xt_train, c_train], y_train,
        batch_size=BATCH_SIZE, epochs=15,  # Reduced for CV speed
        validation_data=([Xp_val, Xt_val, c_val], y_val),
        callbacks=[early_stop], verbose=0
    )
    
    # Evaluate V5
    v5_metrics = evaluate_model(model_v5_cv, Xp_val, Xt_val, y_val, c_val, "v5")
    v5_fold_results.append(v5_metrics)
    print(f"V5 - Hit@5: {v5_metrics['hit@5']:.4f}, MRR: {v5_metrics['mrr']:.4f}")
    
    # Train V5.1
    print("Training V5.1...")
    model_v5_1_cv = build_model_v5_1(
        num_classes=NUM_CLASSES, vocab_size=VOCAB_SIZE,
        max_hist=MAX_ORDER_HISTORY, max_items=MAX_ITEMS_PER_ORDER,
        lstm_units=64, n_cluster=n_clusters, dropout=DROPOUT
    )
    model_v5_1_cv.fit(
        [Xp_train, Xt_train, c_train], 
        {"logits": y_train, "attention_flat": np.zeros((len(y_train), MAX_ORDER_HISTORY))},
        batch_size=BATCH_SIZE, epochs=15,
        validation_data=([Xp_val, Xt_val, c_val], {"logits": y_val, "attention_flat": attn_dummy}),
        callbacks=[early_stop], verbose=0
    )
    
    # Evaluate V5.1
    v5_1_metrics = evaluate_model(model_v5_1_cv, Xp_val, Xt_val, y_val, c_val, "v5.1")
    v5_1_fold_results.append(v5_1_metrics)
    print(f"V5.1 - Hit@5: {v5_1_metrics['hit@5']:.4f}, MRR: {v5_1_metrics['mrr']:.4f}")

K-Fold Cross-Validation

--- Fold 1/5 ---
Train: 6645, Val: 1662
Training V5...
V5 - Hit@5: 0.0463, MRR: 0.0300
Training V5.1...
V5.1 - Hit@5: 0.0367, MRR: 0.0210

--- Fold 2/5 ---
Train: 6645, Val: 1662
Training V5...
V5 - Hit@5: 0.0560, MRR: 0.0335
Training V5.1...
V5.1 - Hit@5: 0.0265, MRR: 0.0168

--- Fold 3/5 ---
Train: 6646, Val: 1661
Training V5...
V5 - Hit@5: 0.0674, MRR: 0.0407
Training V5.1...
V5.1 - Hit@5: 0.0385, MRR: 0.0235

--- Fold 4/5 ---
Train: 6646, Val: 1661
Training V5...
V5 - Hit@5: 0.0680, MRR: 0.0431
Training V5.1...
V5.1 - Hit@5: 0.0452, MRR: 0.0265

--- Fold 5/5 ---
Train: 6646, Val: 1661
Training V5...
V5 - Hit@5: 0.0542, MRR: 0.0309
Training V5.1...
V5.1 - Hit@5: 0.0433, MRR: 0.0249


## 8. Results Summary

In [14]:
print("\n" + "=" * 60)
print("CROSS-VALIDATION RESULTS")
print("=" * 60)

# Aggregate results
def aggregate_results(fold_results):
    """Average metrics across folds."""
    return {
        "hit@1": np.mean([r["hit@1"] for r in fold_results]),
        "hit@3": np.mean([r["hit@3"] for r in fold_results]),
        "hit@5": np.mean([r["hit@5"] for r in fold_results]),
        "hit@10": np.mean([r["hit@10"] for r in fold_results]),
        "mrr": np.mean([r["mrr"] for r in fold_results]),
    }

v5_avg = aggregate_results(v5_fold_results)
v5_1_avg = aggregate_results(v5_1_fold_results)

# Print results table
print(f"\n{'Metric':<12} {'V5':>12} {'V5.1':>12} {'Diff':>12}")
print("-" * 50)
for metric in ["hit@1", "hit@3", "hit@5", "hit@10", "mrr"]:
    v5_val = v5_avg[metric] * 100
    v5_1_val = v5_1_avg[metric] * 100
    diff = v5_1_val - v5_val
    print(f"{metric:<12} {v5_val:>11.1f}% {v5_1_val:>11.1f}% {diff:>+11.1f}%")

# Winner summary
print("\n" + "-" * 50)
if v5_1_avg["hit@5"] > v5_avg["hit@5"]:
    print("WINNER: V5.1 (LSTM + Attention)")
else:
    print("WINNER: V5 (LSTM without Attention)")


CROSS-VALIDATION RESULTS

Metric                 V5         V5.1         Diff
--------------------------------------------------
hit@1                1.4%         0.7%        -0.7%
hit@3                4.1%         2.3%        -1.8%
hit@5                5.8%         3.8%        -2.0%
hit@10              10.8%         7.9%        -2.9%
mrr                  3.6%         2.3%        -1.3%

--------------------------------------------------
WINNER: V5 (LSTM without Attention)


## Summary

This notebook trained and benchmarked both V5 and V5.1 recommendation models:
- **V5**: LSTM without attention (baseline)
- **V5.1**: LSTM with dot-product attention

Cross-validation results show comparative performance.